# Train st1 (Relation Classification) — RoBERTa-large

Second transformer in the multistep CausalSense pipeline.

**Task:** 5-way classification of the relation type — `cause` / `enable` / `intend` / `prevent` / `no_relation`.

**Data:**
- train: `data/Combined_dataset_CommonSense+News_Data/combined.csv` (6,792 rows, all 5 classes)
- dev: `data/News_data/dev.csv` (627 rows)
- test: `data/Test_dataset/test.csv` (632 rows)

**Output:** a HuggingFace-format checkpoint folder (with `id2label` baked into `config.json`).

**Runtime:** ~3–4 hr on a T4 (more training data than st0). Set Runtime → Change runtime type → T4 GPU.

## 0. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies

Adds `datasets` on top of what st0 needed.

In [ ]:
!pip install -q transformers==4.44.2 datasets scikit-learn pandas tqdm

## 2. Clone the repo

Public repo — no auth needed.

In [ ]:
import os

if not os.path.isdir('Relation_extraction'):
    !git clone https://github.com/eitang5/Relation_extraction.git
else:
    print('Repo already cloned.')

%cd Relation_extraction

## 3. Mount Google Drive (recommended)

Checkpoints survive Colab disconnects. Skip and use a local `OUTPUT_DIR` if you don't mind retraining.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/causalsense/checkpoints

## 4. Sanity run — small subset, 1 epoch on roberta-base (~2–3 min)

Trains on 200/50/50 rows for one epoch to confirm the pipeline works end-to-end. Watch for:

- Printed label distribution should show all 5 classes (cause / enable / intend / prevent / no_relation).
- Classification report at the end should have nonzero entries — but **don't expect good numbers** from 200 rows × 1 epoch. The goal is just "no crash."

Skip if you're confident.

In [ ]:
import os
os.environ['MODEL_NAME']  = 'roberta-base'
os.environ['OUTPUT_DIR']  = 'checkpoints/st1_sanity'
os.environ['EPOCHS']      = '1'
os.environ['BATCH_SIZE']  = '16'
os.environ['MAX_TRAIN']   = '200'
os.environ['MAX_DEV']     = '50'
os.environ['MAX_TEST']    = '50'

!python separate_tasks/relation_classification.py

## 5. Real run — roberta-large, 10 epochs, full data (~3–4 hr on T4)

Output goes to Drive. The script saves the best (lowest val-loss) checkpoint.

**Important:** unset the `MAX_*` env vars from the sanity run, or they'll cap the real run too.

In [ ]:
import os
# Clear the caps from the sanity cell
for k in ('MAX_TRAIN', 'MAX_DEV', 'MAX_TEST'):
    os.environ.pop(k, None)

os.environ['MODEL_NAME']  = 'FacebookAI/roberta-large'
os.environ['OUTPUT_DIR']  = '/content/drive/MyDrive/causalsense/checkpoints/st1_roberta_large'
os.environ['EPOCHS']      = '10'
os.environ['BATCH_SIZE']  = '8'

!python separate_tasks/relation_classification.py

## 6. Verify the checkpoint reloads

`id2label` should be populated with the 5 relation names (this is what makes inference output human-readable labels later).

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ckpt = os.environ['OUTPUT_DIR']
model = AutoModelForSequenceClassification.from_pretrained(ckpt)
tokenizer = AutoTokenizer.from_pretrained(ckpt)

print('Loaded:', ckpt)
print('Num labels:', model.config.num_labels)
print('id2label  :', model.config.id2label)
print('Params (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

## 7. Smoke-test inference

One example per class (roughly) to eyeball the predictions.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
id2label = model.config.id2label

samples = [
    ('cause      ', 'The earthquake caused widespread destruction in coastal areas.'),
    ('prevent    ', 'Heavy rain prevented the football match from taking place.'),
    ('intend     ', 'She studied hard in order to pass the exam.'),
    ('enable     ', 'Internet access enables employees to work remotely.'),
    ('no_relation', 'The cat sat on the mat.'),
]

for expected, s in samples:
    inputs = tokenizer(s, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        pred_id = int(logits.argmax(-1).item())
        probs = torch.softmax(logits, -1)[0]
    pred = id2label[pred_id]
    conf = float(probs[pred_id])
    print(f'expected={expected} | pred={pred:>12} ({conf:.2f}) | {s}')

## Next steps

- **st0 + st1 is a complete two-transformer pipeline** for relation-label-only output. At inference: run st0 first, keep sentences it predicts as `1`, run those through st1 to get the relation type.
- Add **st2** (`separate_tasks/EE.py`) only if you also need subject/object event spans.
- **To continue training this checkpoint on your own data later:** set `MODEL_NAME` to the saved folder path, point `TRAIN_FILE` / `DEV_FILE` / `TEST_FILE` at your CSVs (columns: `text`, `relation` with values from {cause, enable, intend, prevent, no_relation}), and re-run.

Example:
```python
os.environ['MODEL_NAME'] = '/content/drive/MyDrive/causalsense/checkpoints/st1_roberta_large'
os.environ['TRAIN_FILE'] = '/content/drive/MyDrive/my_data/train.csv'
os.environ['DEV_FILE']   = '/content/drive/MyDrive/my_data/dev.csv'
os.environ['TEST_FILE']  = '/content/drive/MyDrive/my_data/test.csv'
os.environ['OUTPUT_DIR'] = '/content/drive/MyDrive/causalsense/checkpoints/st1_roberta_large_mydata'
os.environ['EPOCHS']     = '3'
!python separate_tasks/relation_classification.py
```